# Este Notebook contiene los scrips que se pueden ejecutar desde el area

# Procesar las ventas diarias

In [ ]:
# Importamos las librerias a usar
import pandas as pd
from clase_reportes import ReportClass
rc = ReportClass()


# Procesar ventas y notas

In [ ]:
# porcesar base de ventas
ventas_procesadas = rc.transformar_base(origen=True)
ruta = rc.validar_ruta()

ruta_clean = ruta / 'CLEAN DATA' 
ruta_carpeta = ruta_clean / f'VENTAS_{ventas_procesadas['nombre_archivo'].split('/')[6]}'

ventas_procesadas['Base'].to_excel(ruta_carpeta, index=False)


# Consolidar la informacion de Clean Data

In [ ]:
# Consolidar ventas
base_clean = rc.consolidar_carpeta(ruta_carpeta=ruta_clean)
ruta_base = ruta / 'file' / 'BASE VENTAS 2025.xlsx'
import locale
try:
    # Intentamos usar el locale en español para obtener "ENERO", "FEBRERO", etc.
    locale.setlocale(locale.LC_TIME, 'es_ES.UTF-8')
except locale.Error:
    print("   - Advertencia: Locale 'es_ES.UTF-8' no disponible. Se usarán nombres de mes en inglés.")
    
base_clean['MES'] = base_clean['FECHA_FACTURA'].dt.strftime('%B').str.upper()
columnas_finales = [
        "Source.Name", "NUMERO_FACTURA", "FECHA_FACTURA", "AÑO", "MES", "DIA",
        "CLIENTE", "IDENTIFICACION_CLIENTE", "CATEGORÍA", "PRODUCTO", "CANTIDAD",
        "TOTAL", "TASA_CAMBIO", "TRM", "TOTAL($)", "TELEFONO", "EMAIL", "PAIS",
        "CIUDAD", "CIUDAD_CORREGIDA", "DEPARTAMENTO", "EQUIPO_VENTAS", "REFERENCIA"
    ]
    
# Manejo defensivo por si la columna 'ASESOR COMERCIAL' no siempre existe
if 'ASESOR COMERCIAL' in base_clean.columns:
    base_clean['ASESOR COMERCIAL'] = base_clean['ASESOR COMERCIAL'].astype(str)
    columnas_finales.append('ASESOR COMERCIAL')

# Aseguramos que solo reordenamos las columnas que realmente existen en el DataFrame
columnas_existentes = [col for col in columnas_finales if col in base_clean.columns]
base_clean = base_clean[columnas_existentes]

 # Esta linea mantiene solo los pruductos comerciales
base_clean = base_clean[base_clean['PRODUCTO'].str.startswith(('[PCN','[KD','[TNG','[B8'))]   ###### linea modificada
base_clean.to_excel(ruta_base, index=False)
